# LatestSummary v1.2 — Yahoo Consensus Cross-Sectional Analysis

## Purpose
Analyzes the **latest snapshot** of Yahoo Consensus data to rank stocks by target price, revenue estimates, EPS estimates, and analyst agreement/disagreement.

## Input
- Latest `*_YahooConsensus.csv` file in the folder (auto-detected by most recent date)
- Filters to tickers with >= 5 analysts (`MIN_ANALYSTS`)

## Output File
`YYYYMMDD_YahooConsensusAnalysis.xlsx` with **8 sheets**

---

## Sheet Descriptions

### **Base Ranking Sheets (TOP 10 per category)**

**1. TP (Target Price)**  
Ranks stocks by:
- TP appreciation (how far current price is below target)
- TP(Min), TP(Avg), TP(Max) absolute values
- TP dispersion bands (narrow = high agreement, wide = low agreement)

Each metric shows TOP 10 highest and TOP 10 lowest  
Columns: Header, Rank, FIN Yr, YAHOO Ticker, metric values

**2. Revenue**  
Ranks stocks by revenue estimates (Min/Avg/Max) for Y1 and Y2  
Shows TOP 10 highest and TOP 10 lowest per metric  
Use case: Identify high-growth vs low-growth revenue outlooks

**3. EPS**  
Ranks stocks by EPS estimates (Min/Avg/Max) for Y1 and Y2  
Shows TOP 10 highest and TOP 10 lowest per metric  
Use case: Identify earnings growth vs contraction expectations

**4. Bands**  
Groups stocks by analyst agreement level:
- **Narrow bands** — low dispersion: (TP(Max) - TP(Min)) / TP(Avg) < threshold → High conviction, analysts agree  
- **Wide bands** — high dispersion → Low conviction, analysts disagree  

Separate rankings for Revenue and EPS dispersion  
Shows ALL tickers in each band category

---

### **Insight Sheets (ALL tickers)**

**5. Summary**  
Combines all base sheets (TP, Revenue, EPS, Bands) into one master list  
Contains ALL ranked results across all categories  
Columns: Header, Rank, FIN Yr, Ticker  
**Use case:** Source data for all insights below

**6. InsightsFrequency**  
Counts how many times each ticker appears across ALL ranking categories  
- `Frequency` — total appearances (higher = appears in more categories)

**Sorted by:** Frequency DESC  
**Use case:** Stocks appearing in many categories = notable across multiple metrics

**7. InsightsCategorySpread**  
How many category *types* does each ticker appear in?  
Categories counted:
- TP categories (any TP-related ranking)
- Revenue categories (any Rev-related ranking)  
- EPS categories (any EPS-related ranking)
- Bands categories (narrow/wide dispersion)

Shows count per category type + total categories (0-4)  
**Sorted by:** Total categories DESC  
**Use case:** Tickers in all 4 types = comprehensive standout

**8. InsightsRankStrength**  
Sum of `1/Rank` scores across all appearances  
- Rank 1 contributes score of 1.0  
- Rank 10 contributes score of 0.1  
- Total_Score = sum across all categories ticker appears in  

Higher score = consistently high ranks across categories  
**Sorted by:** Score DESC  
**Use case:** Identify stocks that rank well across multiple dimensions (not just one outlier metric)

**9. InsightsYearWise**  
Compares Y1 vs Y2 ranking strength  
- `Score_Y1` — sum of 1/Rank for all Y1-related categories  
- `Score_Y2` — sum of 1/Rank for all Y2-related categories  
- Shows which year has stronger rankings for each ticker  

**Sorted by:** Score_Y1 DESC  
**Use case:** See if consensus is more optimistic about near-term (Y1) or long-term (Y2)

---

## Key Differences vs TrendSummary

| Aspect | LatestSummary | TrendSummary |
|--------|---------------|--------------|
| **Data used** | Latest snapshot only | Time series (all CSVs) |
| **Question answered** | "Who ranks highest *now*?" | "Who is being *revised* up/down?" |
| **Time dimension** | Cross-sectional | Longitudinal |
| **Signals** | Absolute levels, dispersion | Changes over time, momentum |
| **Best for** | Finding outliers, conviction | Finding estimate revision trends |

---

## Key Parameters
- `MIN_ANALYSTS = 5` — minimum analyst count to include ticker  
- `TOP_N` — varies by metric, typically 10-20 per category in base sheets  
- Insights sheets show **ALL tickers** (not limited by TOP_N)

## Methodology
- **Ranking:** Sorted by metric value (ascending or descending depending on category)
- **Dispersion:** (Max - Min) / Avg — lower = higher agreement
- **Bands:** 
  - Narrow = dispersion below median  
  - Wide = dispersion above median  
- **Score calculation:** 1/Rank composite summed across categories

In [2]:
import pandas as pd
import os
import re

# ================= CONFIG =================
FOLDER = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus"
MIN_ANALYSTS = 5
TOP_N = 10

# ================= FIND LATEST FILE =================
files = [f for f in os.listdir(FOLDER) if re.match(r"\d{8}_YahooConsensus\.csv", f)]
latest_file = max(files)
path = os.path.join(FOLDER, latest_file)

print("Using file:", latest_file)

# Extract yyyymmdd for Excel filename
file_date = latest_file.split("_")[0]
excel_output = os.path.join(FOLDER, f"{file_date}_YahooConsensusAnalysis.xlsx")

# ================= LOAD =================
df = pd.read_csv(path)

# ================= PREP =================
df = df[df["#Count"] >= MIN_ANALYSTS]

# Identify Y1 & Y2
fy_sorted = sorted(df["FIN Yr"].unique())
Y1, Y2 = fy_sorted[0], fy_sorted[1]
print("Y1:", Y1, "| Y2:", Y2)

# Keep one row per ticker
df_tp = (
    df.sort_values("FIN Yr")
      .drop_duplicates(subset=["YAHOO Ticker"], keep="first")
)

# Add rank and header
def add_rank_and_header(df, header):
    df2 = df.copy()

    # Remove existing Header or Rank if present
    if "Header" in df2.columns:
        df2 = df2.drop(columns=["Header"])
    if "Rank" in df2.columns:
        df2 = df2.drop(columns=["Rank"])

    # Add Rank
    df2.insert(0, "Rank", range(1, len(df2) + 1))

    # Add Header
    df2.insert(0, "Header", header)

    return df2

# ================= BUILD TABLES (all 3 TP columns) =================
tables = {
    "Top Min Target Price Appreciation": add_rank_and_header(
        df_tp[[
            "YAHOO Ticker",
            "TP_Appreciation_(Min)",
            "TP_Appreciation_(Avg)",
            "TP_Appreciation_(Max)",
            "#Count"
        ]].sort_values("TP_Appreciation_(Min)", ascending=False).head(TOP_N),
        "Top Min Target Price Appreciation"
    ),

    "Top Avg Target Price Appreciation": add_rank_and_header(
        df_tp[[
            "YAHOO Ticker",
            "TP_Appreciation_(Min)",
            "TP_Appreciation_(Avg)",
            "TP_Appreciation_(Max)",
            "#Count"
        ]].sort_values("TP_Appreciation_(Avg)", ascending=False).head(TOP_N),
        "Top Avg Target Price Appreciation"
    ),

    "Top Max Target Price Appreciation": add_rank_and_header(
        df_tp[[
            "YAHOO Ticker",
            "TP_Appreciation_(Min)",
            "TP_Appreciation_(Avg)",
            "TP_Appreciation_(Max)",
            "#Count"
        ]].sort_values("TP_Appreciation_(Max)", ascending=False).head(TOP_N),
        "Top Max Target Price Appreciation"
    ),

    "Bottom Min Target Price Appreciation": add_rank_and_header(
        df_tp[[
            "YAHOO Ticker",
            "TP_Appreciation_(Min)",
            "TP_Appreciation_(Avg)",
            "TP_Appreciation_(Max)",
            "#Count"
        ]].sort_values("TP_Appreciation_(Min)", ascending=True).head(TOP_N),
        "Bottom Min Target Price Appreciation"
    ),

    "Bottom Avg Target Price Appreciation": add_rank_and_header(
        df_tp[[
            "YAHOO Ticker",
            "TP_Appreciation_(Min)",
            "TP_Appreciation_(Avg)",
            "TP_Appreciation_(Max)",
            "#Count"
        ]].sort_values("TP_Appreciation_(Avg)", ascending=True).head(TOP_N),
        "Bottom Avg Target Price Appreciation"
    ),

    "Bottom Max Target Price Appreciation": add_rank_and_header(
        df_tp[[
            "YAHOO Ticker",
            "TP_Appreciation_(Min)",
            "TP_Appreciation_(Avg)",
            "TP_Appreciation_(Max)",
            "#Count"
        ]].sort_values("TP_Appreciation_(Max)", ascending=True).head(TOP_N),
        "Bottom Max Target Price Appreciation"
    ),
}

# ================= COMBINE INTO ONE SHEET =================
combined_rows = []

for header, df_tbl in tables.items():
    combined_rows.append(df_tbl.copy())

final_df = pd.concat(combined_rows, ignore_index=True)

# ================= EXPORT TO EXCEL (overwrite) =================
final_df.to_excel(excel_output, sheet_name="TP", index=False)

print("\nTP Sheet created:", excel_output)


Using file: 20260327_YahooConsensus.csv
Y1: FY26 | Y2: FY27

TP Sheet created: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus\20260327_YahooConsensusAnalysis.xlsx


In [3]:
def top_rank(df, fy, metric, label, group="Revenue"):

    # Select correct columns
    if group == "Revenue":
        cols = [
            "FIN Yr",
            "YAHOO Ticker",
            "Rev_Growth_(Min)",
            "Rev_Growth_(Avg)",
            "Rev_Growth_(Max)",
            "TP_Appreciation_(Min)",
            "TP_Appreciation_(Avg)",
            "TP_Appreciation_(Max)",
            "#Count"
        ]

    elif group == "EPS":
        cols = [
            "FIN Yr",
            "YAHOO Ticker",
            "EPS_Growth_(Min)",
            "EPS_Growth_(Avg)",
            "EPS_Growth_(Max)",
            "TP_Appreciation_(Min)",
            "TP_Appreciation_(Avg)",
            "TP_Appreciation_(Max)",
            "#Count"
        ]

    # Build output
    out = (
        df[df["FIN Yr"] == fy]
        .sort_values(metric, ascending=False)
        .head(TOP_N)[cols]
    )

    # Remove existing Rank/Header if present
    for col in ["Header", "Rank"]:
        if col in out.columns:
            out = out.drop(columns=[col])

    # Add Rank
    out.insert(0, "Rank", range(1, len(out) + 1))

    # Add Header
    out.insert(0, "Header", label)

    return out

# ================= BUILD REVENUE TABLES =================
rev_tables = []

rev_tables.append(top_rank(df, Y1, "Rev_Growth_(Min)", "Top Min Revenue Growth (Y1)"))
rev_tables.append(top_rank(df, Y2, "Rev_Growth_(Min)", "Top Min Revenue Growth (Y2)"))

rev_tables.append(top_rank(df, Y1, "Rev_Growth_(Avg)", "Top Avg Revenue Growth (Y1)"))
rev_tables.append(top_rank(df, Y2, "Rev_Growth_(Avg)", "Top Avg Revenue Growth (Y2)"))

rev_tables.append(top_rank(df, Y1, "Rev_Growth_(Max)", "Top Max Revenue Growth (Y1)"))
rev_tables.append(top_rank(df, Y2, "Rev_Growth_(Max)", "Top Max Revenue Growth (Y2)"))

# ================= BUILD EPS TABLES =================
eps_tables = []

eps_tables.append(top_rank(df, Y1, "EPS_Growth_(Min)", "Top Min EPS Growth (Y1)", group="EPS"))
eps_tables.append(top_rank(df, Y2, "EPS_Growth_(Min)", "Top Min EPS Growth (Y2)", group="EPS"))

eps_tables.append(top_rank(df, Y1, "EPS_Growth_(Avg)", "Top Avg EPS Growth (Y1)", group="EPS"))
eps_tables.append(top_rank(df, Y2, "EPS_Growth_(Avg)", "Top Avg EPS Growth (Y2)", group="EPS"))

eps_tables.append(top_rank(df, Y1, "EPS_Growth_(Max)", "Top Max EPS Growth (Y1)", group="EPS"))
eps_tables.append(top_rank(df, Y2, "EPS_Growth_(Max)", "Top Max EPS Growth (Y2)", group="EPS"))

# Combine into one DataFrame
final_rev_df = pd.concat(rev_tables, ignore_index=True)
final_eps_df = pd.concat(eps_tables, ignore_index=True)

with pd.ExcelWriter(excel_output, engine="openpyxl", mode="w") as writer:
    final_df.to_excel(writer, sheet_name="TP", index=False)
    final_rev_df.to_excel(writer, sheet_name="Revenue", index=False)
    final_eps_df.to_excel(writer, sheet_name="EPS", index=False)

print("\nRevenue & EPS sheet created:", excel_output)


Revenue & EPS sheet created: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus\20260327_YahooConsensusAnalysis.xlsx


In [4]:
def band_rank(df, fy, group="Revenue", narrow=True):
    tmp = df[df["FIN Yr"] == fy].copy()

    if group == "Revenue":
        tmp["Band_Sum"] = tmp["Rev_Max_Avg_Change"] + tmp["Rev_Avg_Min_Change"].abs()
        tmp = tmp[
            (tmp["Band_Sum"] > 0) &
            (tmp["Rev_Growth_(Min)"] < tmp["Rev_Growth_(Avg)"]) &
            (tmp["Rev_Growth_(Avg)"] < tmp["Rev_Growth_(Max)"])
        ]

        # Rename to generic Growth columns
        tmp = tmp.rename(columns={
            "Rev_Growth_(Min)": "Growth_(Min)",
            "Rev_Growth_(Avg)": "Growth_(Avg)",
            "Rev_Growth_(Max)": "Growth_(Max)"
        })

        prefix = "Revenue"

    elif group == "EPS":
        tmp["Band_Sum"] = tmp["EPS_Max_Avg_Change"] + tmp["EPS_Avg_Min_Change"].abs()
        tmp = tmp[
            (tmp["Band_Sum"] > 0) &
            (tmp["EPS_Growth_(Min)"] < tmp["EPS_Growth_(Avg)"]) &
            (tmp["EPS_Growth_(Avg)"] < tmp["EPS_Growth_(Max)"])
        ]

        # Rename to generic Growth columns
        tmp = tmp.rename(columns={
            "EPS_Growth_(Min)": "Growth_(Min)",
            "EPS_Growth_(Avg)": "Growth_(Avg)",
            "EPS_Growth_(Max)": "Growth_(Max)"
        })

        prefix = "EPS"

    # Narrowest = ascending, Broadest = descending
    asc = True if narrow else False

    out = (
        tmp.sort_values("Band_Sum", ascending=asc)
           .head(TOP_N)
           [[
               "FIN Yr",
               "YAHOO Ticker",
               "Growth_(Min)",
               "Growth_(Avg)",
               "Growth_(Max)",
               "Band_Sum",
               "TP_Appreciation_(Min)",
               "TP_Appreciation_(Avg)",
               "TP_Appreciation_(Max)",
               "#Count"
           ]]
    )

    # Remove existing Rank/Header if present
    for col in ["Header", "Rank"]:
        if col in out.columns:
            out = out.drop(columns=[col])

    # Add Rank
    out.insert(0, "Rank", range(1, len(out) + 1))

    # Add Header
    label = f"{prefix} Band - {'Narrowest' if narrow else 'Broadest'} (Y{1 if fy == Y1 else 2})"
    out.insert(0, "Header", label)

    return out

# ================= BUILD BAND TABLES =================
band_tables = []

# Revenue Narrowest
band_tables.append(band_rank(df, Y1, group="Revenue", narrow=True))
band_tables.append(band_rank(df, Y2, group="Revenue", narrow=True))

# Revenue Broadest
band_tables.append(band_rank(df, Y1, group="Revenue", narrow=False))
band_tables.append(band_rank(df, Y2, group="Revenue", narrow=False))

# EPS Narrowest
band_tables.append(band_rank(df, Y1, group="EPS", narrow=True))
band_tables.append(band_rank(df, Y2, group="EPS", narrow=True))

# EPS Broadest
band_tables.append(band_rank(df, Y1, group="EPS", narrow=False))
band_tables.append(band_rank(df, Y2, group="EPS", narrow=False))

final_band_df = pd.concat(band_tables, ignore_index=True)

with pd.ExcelWriter(excel_output, engine="openpyxl", mode="w") as writer:
    final_df.to_excel(writer, sheet_name="TP", index=False)
    final_rev_df.to_excel(writer, sheet_name="Revenue", index=False)
    final_eps_df.to_excel(writer, sheet_name="EPS", index=False)
    final_band_df.to_excel(writer, sheet_name="Bands", index=False)

print("\nBand sheet created:", excel_output)


Band sheet created: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus\20260327_YahooConsensusAnalysis.xlsx


In [5]:
import pandas as pd

# --- Read the four sheets ---
tp_df   = pd.read_excel(excel_output, sheet_name="TP")
rev_df  = pd.read_excel(excel_output, sheet_name="Revenue")
eps_df  = pd.read_excel(excel_output, sheet_name="EPS")
band_df = pd.read_excel(excel_output, sheet_name="Bands")

# --- TP sheet has no FIN Yr → assign Y1 explicitly ---
tp_df["FIN Yr"] = Y1

# --- Function to detect and standardize ticker column ---
def fix_ticker(df):
    # Clean column names: strip spaces, remove non-breaking spaces
    clean_cols = []
    for c in df.columns:
        c2 = c.replace("\xa0", " ")   # remove non-breaking space
        c2 = c2.strip()               # remove leading/trailing spaces
        clean_cols.append(c2)
    df.columns = clean_cols

    # Detect ticker column
    ticker_cols = [c for c in df.columns if "ticker" in c.lower()]

    if not ticker_cols:
        print("DEBUG: Columns found =", df.columns.tolist())
        raise KeyError("No ticker column found in this sheet")

    # Rename first match to standard name
    df = df.rename(columns={ticker_cols[0]: "Ticker"})
    return df

# --- Apply ticker fix to all sheets ---
tp_df   = fix_ticker(tp_df)
rev_df  = fix_ticker(rev_df)
eps_df  = fix_ticker(eps_df)
band_df = fix_ticker(band_df)

# --- Extract required columns ---
tp_small   = tp_df[["Header", "Rank", "FIN Yr", "Ticker"]]
rev_small  = rev_df[["Header", "Rank", "FIN Yr", "Ticker"]]
eps_small  = eps_df[["Header", "Rank", "FIN Yr", "Ticker"]]
band_small = band_df[["Header", "Rank", "FIN Yr", "Ticker"]]

# --- Combine into Summary sheet ---
summary_df = pd.concat(
    [tp_small, rev_small, eps_small, band_small],
    ignore_index=True
)

# --- Write Summary to Excel ---
with pd.ExcelWriter(excel_output, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print("Summary sheet created successfully.")

Summary sheet created successfully.


In [6]:
import pandas as pd

# --- Read the Summary sheet ---
summary_df = pd.read_excel(excel_output, sheet_name="Summary")

# --- Clean column names (remove hidden spaces) ---
summary_df.columns = [c.replace("\xa0", " ").strip() for c in summary_df.columns]

# --- Detect ticker column dynamically ---
ticker_cols = [c for c in summary_df.columns if "ticker" in c.lower()]
if not ticker_cols:
    print("DEBUG: Summary columns =", summary_df.columns.tolist())
    raise KeyError("No ticker column found in Summary sheet")

ticker_col = ticker_cols[0]   # actual column name in Summary

# --- Frequency Analysis ---
freq_df = (
    summary_df[ticker_col]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "Ticker", ticker_col: "Frequency"})
)

# --- Write Insights A to Excel ---
with pd.ExcelWriter(excel_output, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    freq_df.to_excel(writer, sheet_name="InsightsFrequency", index=False)

print("Insights A (Frequency) created successfully.")

Insights A (Frequency) created successfully.


In [7]:
import pandas as pd

# --- Read the Summary sheet ---
summary_df = pd.read_excel(excel_output, sheet_name="Summary")

# --- Clean column names (remove hidden spaces) ---
summary_df.columns = [c.replace("\xa0", " ").strip() for c in summary_df.columns]

# --- Detect ticker column dynamically ---
ticker_cols = [c for c in summary_df.columns if "ticker" in c.lower()]
if not ticker_cols:
    print("DEBUG: Summary columns =", summary_df.columns.tolist())
    raise KeyError("No ticker column found in Summary sheet")

ticker_col = ticker_cols[0]   # actual column name

# --- Category Spread: list all Headers for each Ticker ---
category_df = (
    summary_df.groupby(ticker_col)["Header"]
    .apply(list)
    .reset_index()
    .rename(columns={ticker_col: "Ticker", "Header": "Appears In"})
)

# --- Write Insights B to Excel ---
with pd.ExcelWriter(excel_output, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    category_df.to_excel(writer, sheet_name="InsightsCategorySpread", index=False)

print("Insights B (Category Spread) created successfully.")

Insights B (Category Spread) created successfully.


In [8]:
import pandas as pd

# --- Read the Summary sheet ---
summary_df = pd.read_excel(excel_output, sheet_name="Summary")

# --- Clean column names (remove hidden spaces) ---
summary_df.columns = [c.replace("\xa0", " ").strip() for c in summary_df.columns]

# --- Detect ticker column dynamically ---
ticker_cols = [c for c in summary_df.columns if "ticker" in c.lower()]
if not ticker_cols:
    print("DEBUG: Summary columns =", summary_df.columns.tolist())
    raise KeyError("No ticker column found in Summary sheet")

ticker_col = ticker_cols[0]   # actual column name

# --- Rank Strength Score = sum(1 / Rank) ---
rank_strength_df = (
    summary_df.assign(Score=lambda x: 1 / x["Rank"])
    .groupby(ticker_col)["Score"]
    .sum()
    .reset_index()
    .rename(columns={ticker_col: "Ticker"})
    .sort_values("Score", ascending=False)
)

# --- Write Insights C to Excel ---
with pd.ExcelWriter(excel_output, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    rank_strength_df.to_excel(writer, sheet_name="InsightsRankStrength", index=False)

print("Insights C (Rank Strength Score) created successfully.")

Insights C (Rank Strength Score) created successfully.


In [9]:
import pandas as pd

# --- Read Summary sheet ---
summary_df = pd.read_excel(excel_output, sheet_name="Summary")

# --- Clean column names ---
summary_df.columns = [c.replace("\xa0", " ").strip() for c in summary_df.columns]

# --- Detect ticker column dynamically ---
ticker_cols = [c for c in summary_df.columns if "ticker" in c.lower()]
if not ticker_cols:
    print("DEBUG: Summary columns =", summary_df.columns.tolist())
    raise KeyError("No ticker column found in Summary sheet")

ticker_col = ticker_cols[0]

# --- Split into Y1 and Y2 ---
y1_df = summary_df[summary_df["FIN Yr"] == Y1]
y2_df = summary_df[summary_df["FIN Yr"] == Y2]

# --- Score for Y1 ---
score_y1 = (
    y1_df.assign(Score_Y1=lambda x: 1 / x["Rank"])
    .groupby(ticker_col)["Score_Y1"]
    .sum()
    .reset_index()
)

# --- Score for Y2 ---
score_y2 = (
    y2_df.assign(Score_Y2=lambda x: 1 / x["Rank"])
    .groupby(ticker_col)["Score_Y2"]
    .sum()
    .reset_index()
)

# --- Appearance flags ---
appears_y1 = y1_df[ticker_col].unique()
appears_y2 = y2_df[ticker_col].unique()

# --- Build final table ---
yearwise_df = (
    pd.DataFrame({ "Ticker": summary_df[ticker_col].unique() })
    .merge(score_y1, on="Ticker", how="left")
    .merge(score_y2, on="Ticker", how="left")
)

# Fill missing scores with 0
yearwise_df["Score_Y1"] = yearwise_df["Score_Y1"].fillna(0)
yearwise_df["Score_Y2"] = yearwise_df["Score_Y2"].fillna(0)

# Add appearance flags
yearwise_df["Appears_Y1"] = yearwise_df["Ticker"].isin(appears_y1)
yearwise_df["Appears_Y2"] = yearwise_df["Ticker"].isin(appears_y2)

# --- Write to Excel ---
with pd.ExcelWriter(excel_output, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    yearwise_df.to_excel(writer, sheet_name="InsightsYearWise", index=False)

print("Insights D (Year-wise Strength) created successfully.")

Insights D (Year-wise Strength) created successfully.
